# MeshGraphNet training-budget study

Run on one Kaggle T4 with Internet enabled. Supply the full repository commit SHA and the version of this Kaggle submission. The default stage verifies CUDA, runs the existing smoke study, and measures throughput on the study data.


In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-meshgraphnet-vortex.git"
REF = "SET_TO_FULL_40_CHARACTER_COMMIT_SHA"   # output of `git rev-parse HEAD` for the commit to run
STAGE = "probe"
NOTEBOOK_VERSION = "SET_TO_KAGGLE_VERSION"
SESSION_BUDGET_HOURS = 6.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-meshgraphnet-vortex"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["MGN_NOTEBOOK_VERSION"] = NOTEBOOK_VERSION
assert NOTEBOOK_VERSION != "SET_TO_KAGGLE_VERSION"


In [ ]:
import subprocess
print(subprocess.check_output(["nvidia-smi"], text=True))


In [ ]:
import os, subprocess
def sh(cmd, cwd=None, check=True):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd).returncode

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
print("commit", head)

In [ ]:
sh(f"grep -v '^torch_scatter' requirements.txt > {WORK}/requirements-no-scatter.txt", cwd=REPO)
sh(f"pip install -q -r {WORK}/requirements-no-scatter.txt", cwd=REPO)
sh("pip install -q setuptools wheel && FORCE_ONLY_CPU=1 pip install -q torch_scatter --no-build-isolation", cwd=REPO)
sh('python -c "import torch, physicsnemo, torch_geometric, torch_scatter, tfrecord, sys; assert torch.cuda.is_available(); '
   "out = torch_scatter.scatter(torch.ones(4, 2, device='cuda'), torch.tensor([0, 0, 1, 1], device='cuda'), dim=0, dim_size=2, reduce='sum'); "
   "assert out.is_cuda and out.tolist() == [[2.0, 2.0], [2.0, 2.0]]; "
   "print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, "
   "'| torch_geometric', torch_geometric.__version__, '| torch_scatter', torch_scatter.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)
assert tree_is_clean(), "the installation modified the clone"
sh('python -c "import torch; assert torch.cuda.device_count() == 1; assert \'T4\' in torch.cuda.get_device_name(0)"', cwd=REPO)


In [ ]:
sh("python -m pytest -q", cwd=REPO)

In [ ]:
sh("python scripts/download_data.py --train 200 --valid 10 --test 50 --dry-run", cwd=REPO)
sh("python scripts/download_data.py --train 200 --valid 10 --test 50", cwd=REPO)
sh("cat data/cylinder_flow/manifest.json && df -h /kaggle/working | tail -1", cwd=REPO)

In [ ]:
sh("python scripts/train.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

In [ ]:
import json
import shutil
from pathlib import Path

sh("python scripts/probe.py --config training_budget --matched-data --warmup 50 --steps 300", cwd=REPO)
probe_path = Path(REPO) / "runs/training_budget_probe.json"
probe = json.loads(probe_path.read_text())
probe["notebook_version"] = NOTEBOOK_VERSION
row = probe["per_processor_size"][0]
budgets = (11960, 59800, 119600)
probe["gradient_hours_by_cutoff"] = {step: step / row["steps_per_second"] / 3600 for step in budgets}
probe["projected_hours_extended_seed"] = row["projected_train_hours_per_run"] + 3 * row["projected_eval_hours_per_run"]
probe["projected_hours_three_seed_study"] = 3 * probe["projected_hours_extended_seed"]
Path(WORK, "training_budget_probe.json").write_text(json.dumps(probe, indent=2) + "\n")
shutil.make_archive(str(Path(WORK) / "smoke_cuda"), "zip", Path(REPO) / "runs", "smoke_cuda")
print(json.dumps(probe, indent=2))
